# 2. Create Silver Table

生データの問題点
- salesテーブルのproduct_nameとproductのJSONの情報が一致していない
- 

#### Customers_bronzeからCustomers_silverを作成

- `valid_from`・`valid_to`をDATE型へ変換

In [0]:
CREATE OR REPLACE TABLE customers_silver
AS
SELECT 
  customer_id,
  tax_id,
  tax_code,
  customer_name,
  state,
  city,
  postcode,
  street,
  number,
  unit,
  region,
  district,
  ship_to_address,
  CAST(from_unixtime(valid_from, 'yyyy-MM-dd') AS TIMESTAMP) AS valid_from,
  CAST(from_unixtime(valid_to, 'yyyy-MM-dd') AS TIMESTAMP) AS valid_to,
  units_purchased,
  loyalty_segment,
  current_timestamp() as ingestion_timestamp
FROM customers_bronze;

SELECT * FROM customers_silver;

Databricks visualization. Run in Databricks to view.

---------------

#### sales_bronzeからsales_silverを作成

- `order_date`をTIMESTAMP型へ変換

※これは多分使わない


product_nameとproductが一致していないレコードをフィルターする


--------------------

### sales_orders_bronzeからsilver層のテーブルを作成


#### 一時データとしてtmp_ordersビューを作成する
※サーバーレスのためビューを使用

- `product`カラムのJSONを展開する
- 欠損値の補完
→ `order_datetime`の欠損値を前後の発注の平均値で補完する

In [0]:
CREATE OR REPLACE TEMP VIEW tmp_orders
AS
SELECT
  order_number AS order_id,
  customer_id,
  CAST(
    CASE WHEN order_datetime IS NULL 
      THEN from_unixtime(AVG(order_datetime) OVER (
          ORDER BY order_number
          ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ))
      ELSE from_unixtime(order_datetime)
    END AS TIMESTAMP
  ) AS order_datetime,
  clicked_items,
  col.id AS product_id,
  col.name AS product_name,
  col.price AS price,
  col.curr AS currency,
  col.promotion_info AS promotion_info,
  col.qty AS quantity,
  col.unit AS unit,
  current_timestamp() as ingestion_timestamp
FROM sales_orders_bronze
LATERAL VIEW explode(
  from_json(ordered_products, 'ARRAY<STRUCT<curr:STRING,id:STRING,name:STRING,price:STRING,promotion_info:STRING,qty:STRING,unit:STRING>>')
) AS col;

#### tmp_ordersからproducts_silverを作成

- 商品一覧を管理するテーブル
- ※商品一覧は実際に発注のあった商品`sales_orders_bronze`の`product`カラムから抽出したデータを元に作成しています


In [0]:
CREATE OR REPLACE TABLE products_silver AS
SELECT
  MAX(product_id) AS product_id,
  MAX(product_name) AS product_name,
  MAX(unit) AS unit,
  MAX(ingestion_timestamp) AS ingestion_timestamp
FROM tmp_orders
GROUP BY product_id, product_name;

SELECT * FROM products_silver;

#### tmp_ordersからorder_silverを作成

- `total_price`では1注文あたりの合計価格を求める

In [0]:

CREATE OR REPLACE TABLE orders_silver AS
SELECT 
  order_id,
  MAX(customer_id) AS customer_id,
  SUM(price) AS total_price,
  SUM(quantity) AS total_quantity,
  MAX(order_datetime) AS ordered_at,
  MAX(ingestion_timestamp) AS ingestion_timestamp
FROM tmp_orders
GROUP BY order_id;

SELECT * FROM orders_silver;

#### orders_silverとproducts_silverの中間テーブルを作成する

- orderに関連するproductを記録するテーブル

In [0]:
CREATE OR REPLACE TABLE order_products_silver AS
SELECT
  order_id,
  product_id,
  quantity,
  price,
  promotion_info,
  order_datetime AS ordered_at,
  ingestion_timestamp
FROM tmp_orders;


SELECT * FROM order_products_silver;

orderごとのclicked itemの一覧

#### click_itemsテーブルを作成する

- order確定までにクリックした商品の一覧を保存する

In [0]:
CREATE OR REPLACE TABLE clicked_items_silver 
AS
SELECT
  order_id,
  array_agg(item[0]) AS clicked_items,
  MAX(order_datetime) AS ordered_at,
  MAX(ingestion_timestamp) AS ingestion_timestamp
FROM tmp_orders
LATERAL VIEW explode(from_json(clicked_items, 'ARRAY<ARRAY<STRING>>')) AS item
GROUP BY order_id, customer_id;

SELECT * FROM clicked_items_silver LIMIT 5;

----------------


#### not_purchased_items_silverを作成する

- クリック後に購入されなかった商品を記録する

- 購入した商品の一時ビュー
- ※サーバーレスは一時テーブル不可のためビューを利用


In [0]:
CREATE OR REPLACE TEMP VIEW tmp_purchased_items
AS
SELECT ops.order_id AS order_id, 
  array_agg(p.product_id) AS ordered_items 
FROM order_products_silver ops
JOIN products_silver p
  ON ops.product_id = p.product_id
GROUP BY ops.order_id


- 購入されなかった商品の一覧

In [0]:
CREATE OR REPLACE TABLE not_purchased_items_silver
AS
SELECT pi.order_id,
  array_except(cis.clicked_items, pi.ordered_items) AS not_purchased_items,
  current_timestamp() as ingestion_timestamp
FROM clicked_items_silver cis
JOIN tmp_purchased_items pi
  ON pi.order_id = cis.order_id;

SELECT * FROM not_purchased_items_silver;


---------------------------
---------------------------
これ以下メモ

In [0]:
%sql
SELECT city, max(lon), min(lon), max(lat), min(lat)
FROM customers_bronze
WHERE city IS NOT NULL
GROUP BY city;

In [0]:
%sql
CREATE OR REPLACE VIEW city_bounds AS
SELECT city, max(lon) AS max_lon, min(lon) AS min_lon, max(lat) AS max_lat, min(lat) AS min_lat
FROM customers_bronze
WHERE city IS NOT NULL
GROUP BY city;

In [0]:
%sql
CREATE OR REPLACE FUNCTION get_city_by_address(address STRING)
RETURNS STRING
RETURN (
  concat_ws(',', slice(split(address, ','), 1, 2))
);

In [0]:
select *
from tmp_orders

In [0]:
select *
from products_silver
where product_id = "AVpgIu4Q1cnluZ0-xBK-"